# M01 Toy Models of Superposition


## Toy 设定

把 4 个稀疏概念压进 2 维隐藏空间，观察概念重叠是怎么出现的。


In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(7)

num_features = 4
hidden_dim = 2
encoder = torch.nn.Linear(num_features, hidden_dim, bias=False)
decoder = torch.nn.Linear(hidden_dim, num_features, bias=False)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.05)

for step in range(500):
    active = (torch.rand(512, num_features) < 0.22).float()
    strengths = torch.rand(512, num_features)
    batch = active * strengths
    hidden = encoder(batch)
    recon = decoder(hidden)
    loss = torch.nn.functional.mse_loss(recon, batch) + 0.002 * hidden.abs().mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

samples = torch.eye(num_features)
hidden_samples = encoder(samples).detach()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(hidden_samples[:, 0], hidden_samples[:, 1], s=130, c=range(num_features), cmap="tab10")
for index, point in enumerate(hidden_samples):
    axes[0].annotate(f"f{index}", (point[0].item(), point[1].item()))
axes[0].set_title("Feature positions in 2D hidden space")
axes[0].axhline(0, color="0.8", linewidth=1)
axes[0].axvline(0, color="0.8", linewidth=1)

axes[1].imshow(decoder.weight.detach().T, cmap="coolwarm", aspect="auto")
axes[1].set_title("Decoder weights")
axes[1].set_xlabel("Hidden dimension")
axes[1].set_ylabel("Original feature")
plt.tight_layout()

print("Final loss:", float(loss.detach()))
print("Hidden representations:\n", hidden_samples)


## 小结

一旦隐藏空间小于概念集合，重叠就不再是 bug，而会变成预期中的几何结构。


## 把这本 notebook 变成研究产出

研究工作单 means this notebook is not complete when the cells finish. 请配合 /templates 里的模板，把结果写成可复查的输出。


### 运行前

- 先选一个变量家族：hidden dimension、sparsity 或 loss weight。
- 在做 sweep 之前，先预测几何图像会怎么变。
- 先把 baseline 设置写进 experiment-log 模板。


### 运行后

- 写清这次 run 里概念重叠出现的几何原因。
- 把图直接展示的东西和你对 neuron 的解释分开写。


### 最后交付这些产物

- 1 份带 baseline 和 sweep 的 experiment log。
- 1 张带注释的隐藏空间图。
- 1 份 100-200 字的 memo，说明为什么 neuron semantics 会失效。


## 验收题

- 为什么 superposition 更像一个几何结果，而不只是训练中的 bug？
- 在你的 toy model sweep 里，hidden dimension、稀疏度或 loss 项的变化，哪一个最明显地改变了概念重叠？
- 如果有人坚持把单个 neuron 当成稳定语义单位，你会用这篇文章的哪条证据来反驳？
- 如果你不能在不重看讲义的情况下独立答出其中至少 2 题，就回去重看文章和你的书面输出。
